Homework 2

In [1]:
import numpy as np
import pandas as pd

url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url)

cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = df[cols]
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


Question 1

In [3]:
#There's one column with missing values. What is it?
df.isna().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

Question 2

In [4]:
#What's the median (50% percentile) for variable 'horsepower'?
df['horsepower'].median()

np.float64(254.0)

In [5]:
def prepare_splits(df, seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
    df_test = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)
    return df_train, df_val, df_test

target = 'fuel_efficiency_mpg'
features = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

def get_X(df_part, fillna_value):
    return df_part[features].fillna(fillna_value).values

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    w_full = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    return np.sqrt(se.mean())

Question 3

In [6]:
df_train, df_val, df_test = prepare_splits(df, 42)
y_train = df_train[target].values
y_val = df_val[target].values

X_train_0 = get_X(df_train, 0)
X_val_0 = get_X(df_val, 0)
w0, w = train_linear_regression_reg(X_train_0, y_train, r=0.0)
rmse_0 = rmse(y_val, w0 + X_val_0.dot(w))

mean_hp = df_train['horsepower'].mean()
X_train_mean = get_X(df_train, mean_hp)
X_val_mean = get_X(df_val, mean_hp)
w0m, wm = train_linear_regression_reg(X_train_mean, y_train, r=0.0)
rmse_mean = rmse(y_val, w0m + X_val_mean.dot(wm))

print('RMSE (fill 0):', round(rmse_0, 2))     # 2.21
print('RMSE (fill mean):', round(rmse_mean, 2))  # 2.2

RMSE (fill 0): 2.21
RMSE (fill mean): 2.2


Question 4

In [7]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0r, wr = train_linear_regression_reg(X_train_0, y_train, r=r)
    score = rmse(y_val, w0r + X_val_0.dot(wr))
    print(f'r={r:<6} RMSE={round(score, 3)}')

r=0      RMSE=2.205
r=0.01   RMSE=2.206
r=0.1    RMSE=2.224
r=1      RMSE=2.349
r=5      RMSE=2.409
r=10     RMSE=2.419
r=100    RMSE=2.429


Question 5

In [8]:
scores = []
for seed in range(10):
    dtr, dval, dte = prepare_splits(df, seed)
    ytr = dtr[target].values
    yval = dval[target].values
    Xtr = get_X(dtr, 0)
    Xval = get_X(dval, 0)
    w0s, ws = train_linear_regression_reg(Xtr, ytr, r=0.0)
    scores.append(rmse(yval, w0s + Xval.dot(ws)))

print('std:', round(np.std(scores), 3))

std: 0.029


Question 6

In [9]:
dtr, dval, dte = prepare_splits(df, 9)
df_full_train = pd.concat([dtr, dval]).reset_index(drop=True)

y_full_train = df_full_train[target].values
y_test = dte[target].values
X_full_train = get_X(df_full_train, 0)
X_test = get_X(dte, 0)

w0f, wf = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)
y_pred_test = w0f + X_test.dot(wf)

print('Test RMSE:', round(rmse(y_test, y_pred_test), 3))

Test RMSE: 2.236
